# Llama 3.2 Customer Support Fine-Tuning (QLoRA)

This notebook is used to fine-tune the **Llama 3.2 3B Instruct** model on the **Bitext Customer Support** dataset using **QLoRA (4-bit quantization + LoRA adapters)**.

## Objectives

- Configure the Google Colab environment.
- Load the Llama 3.2 model.
- Apply 4-bit quantization with BitsAndBytes.
- Configure and attach LoRA adapters.
- Fine-tune the model using Supervised Fine-Tuning (SFT).
- Save the trained LoRA adapters for inference.

> **Note**
>
> This notebook is intended for experimentation and training on Google Colab GPUs.
> The production-ready training pipeline is available in the project's `train.py` script.

In [ ]:
#Verify GPU
!nvidia-smi

In [ ]:
# Clone the Project Repository
!git clone https://github.com/andref218/llm_customer_support_finetuning.git

In [ ]:
# Install Dependencies

!pip install -q \
transformers \
datasets \
accelerate \
peft \
huggingface_hub \
bitsandbytes==0.48.2 \
trl==0.25.1

In [ ]:
# Import Libraries

import os
import sys

import torch
import wandb
from google.colab import userdata

# Hugging Face
from huggingface_hub import login
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

# PEFT and TRL
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer, SFTConfig

# Project utilities
sys.path.append("/content/llm_customer_support_finetuning")
from src.formatting import format_example

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

DATASET = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

PROJECT_NAME = "llm_customer_support_finetuning"

#Log in to HuggingFace and Weights & Biases

In [ ]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Log in to Weights & Biases
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

In [ ]:
# Configure 4-bit Quantization with BitsAndBytes
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# Load the Tokenizer and the Model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
print(base_model)

In [ ]:
# Split Dataset into Train, Validation and Test Sets

#The original dataset only provides a training split. We divide it into training, validation, and test sets to
#train the model, monitor its performance during fine-tuning, and evaluate its ability to generalize on unseen data.

train_test = DATASET["train"].train_test_split(
    test_size=0.2,
    seed=42,
)

validation_test = train_test["test"].train_test_split(
    test_size=0.5,
    seed=42,
)

DATASET_SPLIT = DatasetDict(
    {
        "train": train_test["train"],
        "validation": validation_test["train"],
        "test": validation_test["test"],
    }
)

print(DATASET_SPLIT)

In [ ]:
# Format Dataset Using the Llama Chat Template
formatted_dataset = DATASET_SPLIT.map(
    lambda example: format_example(example, tokenizer)
)

print(formatted_dataset)
print(formatted_dataset["train"].column_names)

In [ ]:
# Prepare the quantized model for QLoRA fine-tuning
base_model = prepare_model_for_kbit_training(base_model)

In [ ]:
# Verify the number of trainable parameters after applying LoRA
trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in base_model.parameters())

print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters: {total:,}")

In [ ]:
# Configure LoRA adapters for parameter-efficient fine-tuning
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
# Apply LoRA adapters to the base model
model = get_peft_model(base_model, lora_config)

In [ ]:
# Verify the trainable parameters and inspect the model architecture
model.print_trainable_parameters()
print(model)

In [ ]:
# Configure the training hyperparameters
training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/customer_support_qlora",

    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,

    optim="paged_adamw_32bit",

    fp16=True,
    bf16=False,

    weight_decay=0.001,
    max_grad_norm=0.3,
    warmup_ratio=0.01,
    lr_scheduler_type="cosine",

    logging_steps=10,
    report_to="wandb",
    run_name="qlora-r16-1epoch",

    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    seed=42,

    max_length=512,
)

In [ ]:
# Initialize the SFTTrainer with the model, datasets, and training configuration
trainer = SFTTrainer(
    model=model,
    args=training_args,

    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
)

In [ ]:
# Fine-tune!
trainer.train()

In [ ]:
# Inspect the Fine-Tuned Model Architecture
print(trainer.model)

In [ ]:
# Save LoRA adapters and tokenizer locally
trainer.save_model("./outputs/final_model")
tokenizer.save_pretrained("./outputs/final_model")

In [ ]:
# Save the Fine-Tuned LoRA Adapter in the Drive
trainer.save_model("/content/drive/MyDrive/customer_support_qlora/final_model")
tokenizer.save_pretrained("/content/drive/MyDrive/customer_support_qlora/final_model")

In [ ]:
# Check the size of the saved LoRA adapter directory.
!du -sh ./outputs/final_model

In [ ]:
# ==========================================================
# Inference Helper Function
# ==========================================================

# Generate a response using the fine-tuned LoRA model.
# This helper function formats the user prompt with the
# Llama 3.2 chat template, performs text generation,
# decodes the output, and prints the conversation.

import torch

def ask(question, max_new_tokens=150):
    messages = [
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)

    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    print("=" * 80)
    print(f"User: {question}\n")
    print(f"Assistant:\n{response}")
    print("=" * 80)

    return response

In [ ]:
# Example Inference 1
ask("I want to return my product.")

In [ ]:
# Example Inference 2
ask("My package hasn't arrived.")

In [ ]:
# Example Inference 3
ask("How do I request a refund?")

In [ ]:
# Example Inference 4
ask("Can I change my shipping address?")

In [ ]:
# Example Inference 5
ask("I received the wrong item.")

In [ ]:
# ==========================================================
# Optional: Upload the trained LoRA adapter to the Hugging Face Hub
#
# Uncomment the lines below and replace HF_REPO_ID with your
# own Hugging Face repository if you want to publish the model.
# ==========================================================

# HF_REPO_ID = "your-username/your-model-name"

# trainer.model.push_to_hub(HF_REPO_ID)
# tokenizer.push_to_hub(HF_REPO_ID)